# Eksplorasi Credit Scoring Lab

Notebook opsional. Aplikasi dan training tidak memerlukan Jupyter. Data: UCI Taiwan 2005; unit NTD; target default bulan berikutnya.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "credit_scoring").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from credit_scoring.data import load_dataset, split_dataset
from credit_scoring.config import RAW_FEATURES, TARGET
from credit_scoring.scoring import ScoringEngine

df = load_dataset()
splits = split_dataset(df)
train = df.iloc[splits["train"]]
print("Sumber:", df.shape, "Train:", train.shape)
train.head()


## Kualitas dan distribusi pada train

Gunakan train untuk eksplorasi yang memengaruhi keputusan model. Test tetap menjadi evaluasi setelah pilihan dibekukan.


In [ ]:
train[RAW_FEATURES].isna().sum().to_frame("missing")


In [ ]:
train[TARGET].value_counts(normalize=True).sort_index().plot.bar(title="Proporsi kelas pada train")
plt.xlabel("Default: 0 = tidak, 1 = ya")
plt.ylabel("Proporsi")
plt.show()


In [ ]:
train["LIMIT_BAL"].plot.hist(bins=35, title="Limit kredit pada train (NTD)")
plt.xlabel("NTD")
plt.show()


In [ ]:
train.groupby("PAY_0")[TARGET].agg(["count", "mean"]).rename(columns={"count": "baris", "mean": "proporsi_default"})


## Model yang sama dengan dashboard

Model terlatih disertakan. Untuk melatih ulang jalankan `python -m credit_scoring.train` di terminal root proyek. Jangan menulis preprocessing terpisah untuk inferensi.


In [ ]:
import json
engine = ScoringEngine.load()
payload = json.loads((ROOT / "data/examples/applicant.json").read_text())
result = engine.score(payload)
{k: result[k] for k in ["probability_of_default", "credit_score", "risk_band", "review_flag"]}


In [ ]:
pd.DataFrame(result["explanation"]["factors"])[["label", "log_odds_contribution", "direction"]].head(10)


## Hasil holdout yang sudah dilaporkan

Metrik di bawah dibaca dari laporan training, tanpa memakai test untuk memilih ulang model.


In [ ]:
report = json.loads((ROOT / "reports/metrics.json").read_text())
pd.DataFrame({"Model": report["test"], "Baseline": report["baseline_test"]}).loc[["roc_auc", "pr_auc", "brier", "log_loss", "recall", "precision"]]


## Pertanyaan untuk latihan

1. Apa yang terjadi pada recall dan volume review saat cutoff diturunkan?
2. Mengapa income tidak bisa ditambahkan tanpa sumber data/outcome yang sesuai?
3. Mengapa membuang SEX/AGE belum membuktikan fairness?
4. Mengapa data Taiwan 2005 belum memvalidasi prediksi pada nasabah Indonesia hari ini?
